2) 정확한 방법론: “블랙박스 사각형으로 Homography → 천장 평면 정사영”
Step A) 블랙박스 4꼭짓점 검출

블랙박스는 직사각형이니까,
(1) 마스크(색/명도 기반) → (2) 컨투어 → (3) approxPolyDP로 4점 찾는 방식이 제일 흔함.

잎에 가리는 일이 적고(천장), 대비도 강해서 자동화 가능성이 높아.

Step B) 39×27 cm에 맞춰 “천장 평면으로 워핑”

검출한 4점을 “실제 크기(39×27)”에 대응시키는 Homography(투시변환) 를 구해서,

이미지를 “천장 평면을 위에서 본 것처럼” 펴버려.

이렇게 하면 원근 때문에 찌그러진 길이 문제가 거의 사라짐.

Step C) 워핑된 이미지에서 원기둥카메라 폭/직경을 px로 측정

워핑된 이미지에서는 cm/px가 균일해짐.

원기둥카메라는

가장 간단: bounding box의 너비(px)

더 정확: 타원 피팅(ellipse fit) 해서 장축/단축 중 적절한 값 사용

Step D) 원기둥카메라의 “실제 cm” 산출

예를 들어 워핑 결과가 “1 px = 0.05 cm”이고,

원기둥카메라 폭이 240 px이면 → 12 cm.

이게 네가 원하는 “블랙박스로부터 원기둥카메라 규격 얻기”의 정석 루트야.

In [6]:
pip install opencv-python numpy pandas

In [10]:
%%writefile /content/front_scale_anchor.py
"""정면(45도) 이미지: 블랙박스(39x27cm)로 천장 평면 Homography(워핑) ->
워핑 이미지에서 원통(윗면 카메라) 타원피팅 -> 원통 실제 직경(cm) 추정(앵커) ->
원통만으로 daily scale(cm/px) 산출.

- 목적: '처음이라 최대한 scale 값을 뽑아내도록' 실패 대비 QC/폴백 포함.
- 입력: 블랙박스가 확실히 찍힌 날짜 폴더(이미지 여러 장)
- 출력:
  1) anchor_summary.json : 원통 직경(cm) 레퍼런스(중앙값)
  2) daily_scale.csv : 이미지별 원통 px, cm_per_px_today, QC 플래그
  3) debug/ : 검출 시각화 결과(옵션)

사용법(권장):

필수 설치:
  pip install opencv-python numpy pandas
"""

from __future__ import annotations

import os
import json
import glob
import argparse
from dataclasses import dataclass
from typing import Tuple, Optional, List, Dict

import cv2
import numpy as np
import pandas as pd



# =========================
# Config
# =========================
@dataclass
class Config:
    # Known black box size (cm)
    box_w_cm: float = 39.0
    box_h_cm: float = 27.0

    # Work resolution (fast & consistent)
    work_size: int = 256

    # Search ROIs (relative to resized image)
    # black box tends to be near top-center; adjust if needed
    box_roi_rel: Tuple[float, float, float, float] = (0.35, 0.02, 0.65, 0.35)  # (x1,y1,x2,y2)
    # cylinder camera tends to be near top-left
    cyl_roi_rel: Tuple[float, float, float, float] = (0.00, 0.00, 0.30, 0.25)

    # Threshold parameters
    canny1: int = 60
    canny2: int = 160

    # Filters for black-box quad
    min_box_area_ratio: float = 0.003  # relative to ROI area
    max_box_area_ratio: float = 0.50
    quad_approx_eps: float = 0.04  # larger -> more aggressive simplification

    # Ellipse QC
    min_cyl_area_ratio: float = 0.02  # relative to cylinder ROI area
    max_cyl_area_ratio: float = 0.85
    max_axis_ratio: float = 4.0

    # Debug
    save_debug: bool = True


# =========================
# Utils
# =========================

def ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)


def rel_roi_to_abs(img: np.ndarray, roi_rel: Tuple[float, float, float, float]) -> Tuple[int, int, int, int]:
    h, w = img.shape[:2]
    x1 = int(round(roi_rel[0] * w))
    y1 = int(round(roi_rel[1] * h))
    x2 = int(round(roi_rel[2] * w))
    y2 = int(round(roi_rel[3] * h))
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    return x1, y1, x2, y2


def order_quad_points(pts: np.ndarray) -> np.ndarray:
    """Order quad points as [tl, tr, br, bl]. pts shape: (4,2)"""
    pts = pts.astype(np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.stack([tl, tr, br, bl], axis=0)


def safe_imread(path: str) -> Optional[np.ndarray]:
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        return None
    return img


def resize_to_work(img: np.ndarray, work: int) -> np.ndarray:
    return cv2.resize(img, (work, work), interpolation=cv2.INTER_AREA)


# =========================
# (A) Black box 4-corner detection
# =========================

def detect_blackbox_quad(img256: np.ndarray, cfg: Config) -> Tuple[Optional[np.ndarray], Dict]:
    """Return quad points in full-image coordinates (256 space), ordered tl,tr,br,bl."""
    x1, y1, x2, y2 = rel_roi_to_abs(img256, cfg.box_roi_rel)
    roi = img256[y1:y2, x1:x2]
    roi_area = max(1, roi.shape[0] * roi.shape[1])

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(blur, cfg.canny1, cfg.canny2)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)

    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_score = -1

    for c in cnts:
        area = cv2.contourArea(c)
        if area < cfg.min_box_area_ratio * roi_area or area > cfg.max_box_area_ratio * roi_area:
            continue

        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, cfg.quad_approx_eps * peri, True)
        if len(approx) != 4:
            continue

        pts = approx.reshape(4, 2).astype(np.float32)
        # Score: prefer large area + near-rectangular (cosine of angles)
        rect = cv2.minAreaRect(pts)
        (cx, cy), (rw, rh), ang = rect
        if rw <= 0 or rh <= 0:
            continue
        aspect = max(rw, rh) / max(1e-6, min(rw, rh))
        # Black box is 39x27 => aspect ~1.444
        aspect_target = cfg.box_w_cm / cfg.box_h_cm
        aspect_score = 1.0 - min(1.0, abs(aspect - aspect_target) / aspect_target)
        score = area * (0.3 + 0.7 * aspect_score)

        if score > best_score:
            best_score = score
            best = pts

    meta = {
        "roi_abs": [x1, y1, x2, y2],
        "n_contours": len(cnts),
        "best_score": float(best_score),
    }

    if best is None:
        return None, meta

    best = order_quad_points(best)
    # shift to full-image coords
    best[:, 0] += x1
    best[:, 1] += y1

    meta["quad"] = best.tolist()
    return best, meta


# =========================
# (B) Homography warp to ceiling plane
# =========================

def warp_to_ceiling(img256: np.ndarray, quad: np.ndarray, cfg: Config) -> Tuple[np.ndarray, Dict]:
    """Warp image so that black box becomes axis-aligned with known pixel dimensions."""

    # Choose a target pixel size for the black box on warped image.
    # We keep it reasonably large inside 256 canvas; use 160px width by default.
    target_box_w_px = 160
    target_box_h_px = int(round(target_box_w_px * (cfg.box_h_cm / cfg.box_w_cm)))

    # Destination points for the black box
    dst = np.array(
        [
            [0, 0],
            [target_box_w_px - 1, 0],
            [target_box_w_px - 1, target_box_h_px - 1],
            [0, target_box_h_px - 1],
        ],
        dtype=np.float32,
    )

    H = cv2.getPerspectiveTransform(quad.astype(np.float32), dst)

    # Warp into a canvas same size as input (256x256)
    warped = cv2.warpPerspective(img256, H, (cfg.work_size, cfg.work_size), flags=cv2.INTER_LINEAR)

    # cm per px on warped plane (approximately uniform)
    cm_per_px_x = cfg.box_w_cm / target_box_w_px
    cm_per_px_y = cfg.box_h_cm / target_box_h_px
    cm_per_px = float((cm_per_px_x + cm_per_px_y) / 2.0)

    meta = {
        "target_box_w_px": int(target_box_w_px),
        "target_box_h_px": int(target_box_h_px),
        "cm_per_px_x": float(cm_per_px_x),
        "cm_per_px_y": float(cm_per_px_y),
        "cm_per_px": cm_per_px,
        "H": H.tolist(),
    }

    return warped, meta


# =========================
# (C) Cylinder width extraction in warped image
# =========================

def detect_cylinder_ellipse_px(img256_or_warped: np.ndarray, cfg: Config) -> Tuple[Optional[Dict], Dict]:
    """Detect cylinder camera as ellipse in ROI. Return axis lengths (px) and chosen diameter_px."""
    x1, y1, x2, y2 = rel_roi_to_abs(img256_or_warped, cfg.cyl_roi_rel)
    roi = img256_or_warped[y1:y2, x1:x2]
    roi_area = max(1, roi.shape[0] * roi.shape[1])

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Since cylinder is bright on darker ceiling, try adaptive threshold
    thr = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        -5,
    )
    thr = cv2.morphologyEx(thr, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)

    cnts, _ = cv2.findContours(thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_score = -1

    for c in cnts:
        area = cv2.contourArea(c)
        if area < cfg.min_cyl_area_ratio * roi_area or area > cfg.max_cyl_area_ratio * roi_area:
            continue
        if len(c) < 5:
            continue
        ell = cv2.fitEllipse(c)
        (cx, cy), (ax1, ax2), ang = ell  # axis lengths in px
        major = max(ax1, ax2)
        minor = min(ax1, ax2)
        if minor <= 1e-6:
            continue
        axis_ratio = major / minor
        if axis_ratio > cfg.max_axis_ratio:
            continue

        # Score: prefer larger and more circular
        circ_score = 1.0 - min(1.0, abs(axis_ratio - 1.0))
        score = area * (0.4 + 0.6 * circ_score)
        if score > best_score:
            best_score = score
            best = {
                "ellipse": {
                    "center": [float(cx + x1), float(cy + y1)],
                    "axes": [float(ax1), float(ax2)],
                    "angle": float(ang),
                },
                "major_px": float(major),
                "minor_px": float(minor),
                "diameter_px": float(major),  # choose major as diameter proxy
                "score": float(score),
                "roi_abs": [x1, y1, x2, y2],
            }

    meta = {
        "roi_abs": [x1, y1, x2, y2],
        "n_contours": len(cnts),
        "best_score": float(best_score),
    }

    return best, meta


# =========================
# Debug drawing
# =========================

def draw_debug(
    img256: np.ndarray,
    quad: Optional[np.ndarray],
    cyl: Optional[Dict],
    cfg: Config,
    out_path: str,
    title: str = "",
) -> None:
    vis = img256.copy()
    # ROIs
    bx1, by1, bx2, by2 = rel_roi_to_abs(vis, cfg.box_roi_rel)
    cx1, cy1, cx2, cy2 = rel_roi_to_abs(vis, cfg.cyl_roi_rel)
    cv2.rectangle(vis, (bx1, by1), (bx2, by2), (0, 255, 255), 1)
    cv2.rectangle(vis, (cx1, cy1), (cx2, cy2), (255, 255, 0), 1)

    if quad is not None:
        q = quad.astype(int)
        cv2.polylines(vis, [q], True, (0, 0, 255), 2)
        for i, (x, y) in enumerate(q):
            cv2.circle(vis, (int(x), int(y)), 3, (0, 0, 255), -1)
            cv2.putText(vis, str(i), (int(x) + 2, int(y) - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

    if cyl is not None and "ellipse" in cyl:
        (cx, cy) = cyl["ellipse"]["center"]
        (ax1, ax2) = cyl["ellipse"]["axes"]
        ang = cyl["ellipse"]["angle"]
        cv2.ellipse(vis, (int(cx), int(cy)), (int(ax1 / 2), int(ax2 / 2)), ang, 0, 360, (0, 255, 0), 2)
        cv2.circle(vis, (int(cx), int(cy)), 2, (0, 255, 0), -1)
        cv2.putText(vis, f"diam_px={cyl['diameter_px']:.1f}", (10, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)

    if title:
        cv2.putText(vis, title, (10, 250), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    cv2.imencode(".png", vis)[1].tofile(out_path)


# =========================
# Main pipeline
# =========================

def process_anchor_folder(img_dir: str, out_dir: str, cfg: Config) -> None:
    ensure_dir(out_dir)
    debug_dir = os.path.join(out_dir, "debug")
    if cfg.save_debug:
        ensure_dir(debug_dir)

    # Collect images
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
    paths = []
    for e in exts:
        paths.extend(glob.glob(os.path.join(img_dir, e)))
    paths = sorted(paths)
    if not paths:
        raise FileNotFoundError(f"No images found in: {img_dir}")

    rows = []
    cyl_cm_estimates = []

    for idx, p in enumerate(paths):
        fn = os.path.basename(p)
        img = safe_imread(p)
        if img is None:
            rows.append({"filename": fn, "status": "read_fail"})
            continue

        img256 = resize_to_work(img, cfg.work_size)

        # (A) black box quad
        quad, meta_box = detect_blackbox_quad(img256, cfg)
        box_ok = quad is not None

        warped = None
        meta_warp = None
        cyl_on_warp = None
        cyl_meta = None

        if box_ok:
            # (B) warp
            warped, meta_warp = warp_to_ceiling(img256, quad, cfg)
            # (C) cylinder on warped image
            cyl_on_warp, cyl_meta = detect_cylinder_ellipse_px(warped, cfg)

            if cyl_on_warp is not None:
                # cylinder cm estimate for this image
                cylinder_cm = float(cyl_on_warp["diameter_px"] * meta_warp["cm_per_px"])
                cyl_cm_estimates.append(cylinder_cm)

        # debug
        if cfg.save_debug:
            if box_ok:
                draw_debug(img256, quad, None, cfg, os.path.join(debug_dir, f"{idx:04d}_A_box_{fn}.png"), title="A:box")
                if warped is not None:
                    # show cylinder on warped
                    draw_debug(warped, None, cyl_on_warp, cfg, os.path.join(debug_dir, f"{idx:04d}_C_warp_cyl_{fn}.png"), title="C:warp+cyl")
            else:
                draw_debug(img256, None, None, cfg, os.path.join(debug_dir, f"{idx:04d}_A_box_fail_{fn}.png"), title="A:box_fail")

        rows.append({
            "filename": fn,
            "box_ok": bool(box_ok),
            "box_score": meta_box.get("best_score", None),
            "warp_cm_per_px": meta_warp.get("cm_per_px", None) if meta_warp else None,
            "cyl_ok_on_warp": bool(cyl_on_warp is not None),
            "cyl_diam_px_on_warp": cyl_on_warp.get("diameter_px", None) if cyl_on_warp else None,
            "cyl_cm_est_on_warp": (cyl_on_warp["diameter_px"] * meta_warp["cm_per_px"]) if (cyl_on_warp and meta_warp) else None,
        })

    # 1) derive reference cylinder cm (median)
    if len(cyl_cm_estimates) == 0:
        raise RuntimeError(
            "Failed to estimate cylinder(cm) from anchor folder. "
            "Try adjusting box_roi_rel/cyl_roi_rel or threshold parameters."
        )

    cylinder_cm_ref = float(np.median(np.array(cyl_cm_estimates)))

    anchor_summary = {
        "img_dir": img_dir,
        "n_images": len(paths),
        "n_cylinder_cm_estimates": len(cyl_cm_estimates),
        "cylinder_cm_ref_median": cylinder_cm_ref,
        "cylinder_cm_estimates": [float(x) for x in cyl_cm_estimates[:200]],  # cap for file size
        "notes": {
            "box_size_cm": [cfg.box_w_cm, cfg.box_h_cm],
            "work_size": cfg.work_size,
            "box_roi_rel": cfg.box_roi_rel,
            "cyl_roi_rel": cfg.cyl_roi_rel,
            "diameter_definition": "ellipse major axis on warped ceiling plane",
        },
    }

    with open(os.path.join(out_dir, "anchor_summary.json"), "w", encoding="utf-8") as f:
        json.dump(anchor_summary, f, ensure_ascii=False, indent=2)

    # 2) daily scale for all images using cylinder only (no warp required)
    daily_rows = []
    for idx, p in enumerate(paths):
        fn = os.path.basename(p)
        img = safe_imread(p)
        if img is None:
            daily_rows.append({"filename": fn, "status": "read_fail"})
            continue

        img256 = resize_to_work(img, cfg.work_size)
        cyl, meta_cyl = detect_cylinder_ellipse_px(img256, cfg)

        if cyl is None:
            daily_rows.append({
                "filename": fn,
                "cyl_ok": False,
                "cyl_diam_px": None,
                "cm_per_px_today": None,
                "qc_flag": "cyl_fail",
            })
            if cfg.save_debug:
                draw_debug(img256, None, None, cfg, os.path.join(debug_dir, f"{idx:04d}_D_daily_cyl_fail_{fn}.png"), title="D:cyl_fail")
            continue

        cm_per_px_today = float(cylinder_cm_ref / max(1e-6, cyl["diameter_px"]))

        daily_rows.append({
            "filename": fn,
            "cyl_ok": True,
            "cyl_diam_px": float(cyl["diameter_px"]),
            "cm_per_px_today": cm_per_px_today,
            "qc_flag": "ok",
        })

        if cfg.save_debug:
            draw_debug(img256, None, cyl, cfg, os.path.join(debug_dir, f"{idx:04d}_D_daily_cyl_{fn}.png"), title=f"D:cm/px={cm_per_px_today:.4f}")

    df_anchor = pd.DataFrame(rows)
    df_anchor.to_csv(os.path.join(out_dir, "anchor_pass.csv"), index=False, encoding="utf-8-sig")

    df_daily = pd.DataFrame(daily_rows)
    df_daily.to_csv(os.path.join(out_dir, "daily_scale.csv"), index=False, encoding="utf-8-sig")

    print("\n[Done]")
    print(f"- anchor_summary.json saved: {os.path.join(out_dir, 'anchor_summary.json')}")
    print(f"- anchor_pass.csv saved:    {os.path.join(out_dir, 'anchor_pass.csv')}")
    print(f"- daily_scale.csv saved:    {os.path.join(out_dir, 'daily_scale.csv')}")
    if cfg.save_debug:
        print(f"- debug images:             {debug_dir}")
    print(f"- cylinder_cm_ref (median): {cylinder_cm_ref:.3f} cm")


def parse_args() -> argparse.Namespace:
    ap = argparse.ArgumentParser()
    ap.add_argument("--img_dir", type=str, required=True, help="블랙박스가 찍힌 날짜 폴더")
    ap.add_argument("--out_dir", type=str, required=True, help="출력 폴더")
    ap.add_argument("--no_debug", action="store_true", help="debug 이미지 저장 끄기")
    return ap.parse_args()


if __name__ == "__main__":
    args = parse_args()

    cfg = Config()
    if args.no_debug:
        cfg.save_debug = False

    process_anchor_folder(args.img_dir, args.out_dir, cfg)


Overwriting /content/front_scale_anchor.py


In [11]:
!python /content/front_scale_anchor.py \
--img_dir "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/251213-251226/251213" \
--out_dir "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. Calibrated/251213"


[Done]
- anchor_summary.json saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. Calibrated/251213/anchor_summary.json
- anchor_pass.csv saved:    /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. Calibrated/251213/anchor_pass.csv
- daily_scale.csv saved:    /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. Calibrated/251213/daily_scale.csv
- debug images:             /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. Calibrated/251213/debug
- cylinder_cm_ref (median): 7.557 cm


### 인제 전체 날짜 돌면서 scale_map 뽑아 내기

In [12]:
"""정면 이미지 전수(또는 샘플)에서 '원통카메라(천장)'만 이용해 daily scale(cm/px) 추출.

전제
- anchor 단계에서 이미 cylinder_cm_ref_median(원통카메라 실제 직경 cm)을 확보했다.
  예: 7.557 cm (anchor_summary.json 안에 저장됨)

목표
- 블랙박스/워핑 없이 (D)만 수행한다.
- 지정한 루트 폴더 하위까지 재귀적으로 이미지 탐색한다.
- 860장 같은 소량에서도 '최대한 전부 scale_map 생성'이 목적.
  => 실패하면 fallback(바운딩박스 폭)까지 적용해 성공률을 끌어올린다.

출력
- daily_scale_all.csv : 이미지별 cm_per_px_today, 원통 직경(px), QC 플래그

사용법(Colab/로컬 공통)
1) 아래 '=== 실행부 ==='에서 경로 3개만 수정
2) 실행

필수 설치
  pip install opencv-python numpy pandas
"""

from __future__ import annotations

import os
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Tuple, Optional, Dict, List

import cv2
import numpy as np
import pandas as pd


# =========================
# 설정값(필요시 아주 최소만 수정)
# =========================
WORK_SIZE = 256

# 원통카메라 ROI (256 기준 상대좌표): (x1,y1,x2,y2)
# 현재 데이터 기준으로 좌상단 천장에 원통이 있어서 이 값이 잘 맞는 편.
CYL_ROI_REL: Tuple[float, float, float, float] = (0.00, 0.00, 0.30, 0.25)

# 타원 피팅 QC
MIN_AREA_RATIO = 0.02
MAX_AREA_RATIO = 0.85
MAX_AXIS_RATIO = 4.0

# fallback용: 밝은 물체 threshold 후 bbox 폭을 직경 proxy로 사용
FALLBACK_MIN_AREA_RATIO = 0.01


In [13]:


# =========================
# IO 유틸
# =========================

def safe_imread(path: str) -> Optional[np.ndarray]:
    """Windows/한글 경로 안전 읽기"""
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    return img


def resize_to_work(img: np.ndarray, size: int = WORK_SIZE) -> np.ndarray:
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)


def rel_roi_to_abs(img: np.ndarray, roi_rel: Tuple[float, float, float, float]) -> Tuple[int, int, int, int]:
    h, w = img.shape[:2]
    x1 = int(round(roi_rel[0] * w))
    y1 = int(round(roi_rel[1] * h))
    x2 = int(round(roi_rel[2] * w))
    y2 = int(round(roi_rel[3] * h))
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    return x1, y1, x2, y2


def list_images_recursive(root_dir: str) -> List[str]:
    """지정 폴더 하위까지 전부 탐색"""
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    out: List[str] = []
    for dp, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(exts):
                out.append(os.path.join(dp, f))
    return sorted(out)


def infer_date_from_path(path: str) -> str:
    """경로에 6자리 날짜폴더(예: 251213)가 있으면 date로 사용"""
    parts = os.path.normpath(path).split(os.sep)
    for p in reversed(parts):
        if len(p) == 6 and p.isdigit():
            return p
    return ""


# =========================
# 원통카메라 검출(1) 타원 피팅 (주)
# =========================

def detect_cylinder_by_ellipse(img256: np.ndarray,
                              cyl_roi_rel: Tuple[float, float, float, float] = CYL_ROI_REL,
                              min_area_ratio: float = MIN_AREA_RATIO,
                              max_area_ratio: float = MAX_AREA_RATIO,
                              max_axis_ratio: float = MAX_AXIS_RATIO) -> Optional[Dict]:
    x1, y1, x2, y2 = rel_roi_to_abs(img256, cyl_roi_rel)
    roi = img256[y1:y2, x1:x2]
    roi_area = max(1, roi.shape[0] * roi.shape[1])

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    thr = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        -5,
    )
    thr = cv2.morphologyEx(thr, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)

    cnts, _ = cv2.findContours(thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_score = -1.0

    for c in cnts:
        area = cv2.contourArea(c)
        if area < min_area_ratio * roi_area or area > max_area_ratio * roi_area:
            continue
        if len(c) < 5:
            continue

        (cx, cy), (ax1, ax2), ang = cv2.fitEllipse(c)
        major = max(ax1, ax2)
        minor = min(ax1, ax2)
        if minor <= 1e-6:
            continue
        axis_ratio = major / minor
        if axis_ratio > max_axis_ratio:
            continue

        # 큰데 + 원형에 가까운 걸 선호
        circ_score = 1.0 - min(1.0, abs(axis_ratio - 1.0))
        score = float(area) * (0.4 + 0.6 * circ_score)

        if score > best_score:
            best_score = score
            best = {
                "diameter_px": float(major),
                "score": float(score),
                "method": "ellipse",
            }

    return best


# =========================
# 원통카메라 검출(2) fallback: 밝은 blob bbox 폭
# =========================

def detect_cylinder_by_bbox_fallback(img256: np.ndarray,
                                    cyl_roi_rel: Tuple[float, float, float, float] = CYL_ROI_REL,
                                    min_area_ratio: float = FALLBACK_MIN_AREA_RATIO) -> Optional[Dict]:
    x1, y1, x2, y2 = rel_roi_to_abs(img256, cyl_roi_rel)
    roi = img256[y1:y2, x1:x2]
    roi_area = max(1, roi.shape[0] * roi.shape[1])

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # 밝은 물체(원통) 잡기
    _, thr = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thr = cv2.morphologyEx(thr, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)

    cnts, _ = cv2.findContours(thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_area = -1.0

    for c in cnts:
        area = float(cv2.contourArea(c))
        if area < min_area_ratio * roi_area:
            continue
        x, y, w, h = cv2.boundingRect(c)
        # 폭/높이 둘 중 더 큰걸 직경 proxy로
        diam = float(max(w, h))
        if area > best_area:
            best_area = area
            best = {
                "diameter_px": diam,
                "score": area,
                "method": "bbox",
            }

    return best


# =========================
# 단일 이미지 처리
# =========================

def process_one_image(path: str, cylinder_cm_ref: float) -> Dict:
    date = infer_date_from_path(path)

    img = safe_imread(path)
    if img is None:
        return {
            "path": path, "date": date,
            "cyl_ok": False, "cyl_diam_px": None,
            "cm_per_px_today": None, "qc_flag": "read_fail",
            "method": None,
        }

    img256 = resize_to_work(img)

    det = detect_cylinder_by_ellipse(img256)
    if det is None:
        det = detect_cylinder_by_bbox_fallback(img256)

    if det is None:
        return {
            "path": path, "date": date,
            "cyl_ok": False, "cyl_diam_px": None,
            "cm_per_px_today": None, "qc_flag": "cyl_fail",
            "method": None,
        }

    diam_px = float(det["diameter_px"])
    cm_per_px = float(cylinder_cm_ref / max(1e-6, diam_px))

    return {
        "path": path,
        "date": date,
        "cyl_ok": True,
        "cyl_diam_px": diam_px,
        "cm_per_px_today": cm_per_px,
        "qc_flag": "ok",
        "method": det.get("method", "?")
    }


# =========================
# 메인 실행 함수
# =========================

def run_daily_scale_all(anchor_summary_json: str,
                        front_root_dir: str,
                        out_csv: str,
                        workers: int = 8) -> pd.DataFrame:

    with open(anchor_summary_json, "r", encoding="utf-8") as f:
        anchor = json.load(f)

    cylinder_cm_ref = float(anchor["cylinder_cm_ref_median"])

    paths = list_images_recursive(front_root_dir)
    print(f"Found {len(paths)} images under: {front_root_dir}")
    print(f"Using cylinder_cm_ref_median = {cylinder_cm_ref:.3f} cm")

    rows: List[Dict] = []

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(process_one_image, p, cylinder_cm_ref) for p in paths]
        for fut in as_completed(futs):
            rows.append(fut.result())

    df = pd.DataFrame(rows)
    df = df.sort_values(["date", "path"]).reset_index(drop=True)

    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # 간단 요약
    n = len(df)
    ok = int((df["qc_flag"] == "ok").sum())
    fail = n - ok
    print(f"Saved: {out_csv}")
    print(f"OK: {ok}/{n}  Fail: {fail}/{n}")

    # 실패 샘플 일부 출력
    if fail > 0:
        print("\n[Fail examples]")
        print(df[df["qc_flag"] != "ok"].head(10)[["path", "qc_flag"]])

    return df




In [15]:
# =========================
# === 실행부 ===
# =========================
if __name__ == "__main__":
    # 1) anchor 단계에서 생성된 json (여기서 7.557cm 읽어옴)
    ANCHOR_SUMMARY_JSON = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. Calibrated/251213/anchor_summary.json"  # <- 너 경로로 수정

    # 2) daily scale 뽑을 정면 이미지 루트 폴더 (하위까지 전부 탐색)
    FRONT_ROOT_DIR = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/251213-251226"  # <- 너 경로로 수정

    # 3) 결과 csv 저장 경로
    OUT_CSV = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/251213-251226/260104_daily_scale_to251226.csv"  # <- 너 경로로 수정

    # 4) 병렬 워커 수 (Colab이면 8~16 권장)
    WORKERS = 12

    run_daily_scale_all(
        anchor_summary_json=ANCHOR_SUMMARY_JSON,
        front_root_dir=FRONT_ROOT_DIR,
        out_csv=OUT_CSV,
        workers=WORKERS,
    )


Found 911 images under: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/251213-251226
Using cylinder_cm_ref_median = 7.557 cm
Saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/251213-251226/260104_daily_scale_to251226.csv
OK: 911/911  Fail: 0/911
